In [ ]:
from pathlib import Path

import hvplot.polars  # noqa: F401
import numpy as np
import polars as pl
from scipy import stats

In [ ]:
data_dir = Path("../../data-generation/data/raw/DRAFT002")
data_glob = str(data_dir / "*/rookies.csv")

# Now that we are 100% sure they are UTF-8, scan_csv will work perfectly.
df_rookies = (
    pl.scan_csv(
        data_glob,
        # No encoding argument needed now, defaults to utf8
        infer_schema_length=1000,
        include_file_paths="filepath",
    )
    .with_columns(
        pl.col("filepath").str.extract(r"DRAFT002/(\d+)/", 1).cast(pl.Int32).alias("Year")
    )
    .drop("filepath")
    .collect()
)

print(f"Loaded {df_rookies.shape[0]} rookies across {df_rookies['Year'].n_unique()} years.")

In [ ]:
# Search for any rows containing the replacement character (�)
# This checks all string columns simultaneously.
corrupted_rows = df_rookies.filter(pl.any_horizontal(pl.col(pl.String).str.contains("\ufffd")))

print(f"Found {corrupted_rows.shape[0]} rows with replaced characters.")
if not corrupted_rows.is_empty():
    print(corrupted_rows.select(["Year", "Last_Name", "First_Name", "College"]).head(10))

In [ ]:
df_rookies.sort(by="Grade", descending=True)

In [ ]:
df_rookies.select("Grade").describe()

In [ ]:
score_threshold = 65
proportion_over_threshold = (
    df_rookies.select("Grade").filter(pl.col("Grade") > score_threshold).shape[0]
    / df_rookies.shape[0]
)
print(f"Rookies with grade greater than {score_threshold}: {proportion_over_threshold:.2%}")

In [ ]:
df_rookies.select("Grade").hvplot.hist(y="Grade", bins="auto")

In [ ]:
stats.normaltest(df_rookies.select("Grade"))

In [ ]:
def render_qq_plot(df: pl.DataFrame, col: str):
    # Calculate theoretical quantiles using SciPy
    # probplot returns: ((osm, osr), (slope, intercept, r))
    # osm: theoretical quantiles (z-scores)
    # osr: ordered responses (your actual data sorted)
    (osm, osr), (slope, intercept, _) = stats.probplot(df[col], dist="norm")

    # Build a Polars DataFrame for the plot points
    qq_df = pl.DataFrame({"theoretical": osm, "sample": osr})

    # Create the scatter plot
    scatter = qq_df.hvplot.scatter(
        x="theoretical",
        y="sample",
        title=f"Q-Q Plot: {col}",
        xlabel="Theoretical Quantiles (Normal)",
        ylabel="Sample Quantiles",
        color="#3498db",
        alpha=0.5,
        size=15,
    )

    # Create the reference line (y = mx + b)
    # We use the min/max of the theoretical quantiles for the x-axis
    line_x = [osm.min(), osm.max()]
    line_y = [slope * x + intercept for x in line_x]

    line_df = pl.DataFrame({"x": line_x, "y": line_y})
    line = line_df.hvplot.line(
        x="x", y="y", color="red", line_dash="dashed", label="Normal Reference"
    )

    return scatter * line

In [ ]:
plot = render_qq_plot(df_rookies, "Grade")
plot

In [ ]:
import polars as pl
import scipy.stats as stats


def find_best_distribution_constrained(
    df: pl.DataFrame, col_name: str, floor: float = None, ceiling: float = None
):
    data = df[col_name].to_numpy()

    fixed_params = {}
    if floor is not None:
        fixed_params["floc"] = float(floor)  # Ensure float
    if floor is not None and ceiling is not None:
        fixed_params["fscale"] = float(ceiling - floor)

    candidates = {
        "Beta": stats.beta,
        "Skew-Normal": stats.skewnorm,
        "Gamma": stats.gamma,
        "Log-Normal": stats.lognorm,
    }

    results = []

    for name, dist in candidates.items():
        try:
            params = dist.fit(data, **fixed_params)

            # Calculate Log-Likelihood
            log_pdf = dist.logpdf(data, *params)
            nllf = -np.sum(log_pdf)

            # Calculate AIC: 2k - 2ln(L)
            k = len(params) - len(fixed_params)
            aic = 2 * k + 2 * nllf

            # THE FIX: Cast everything to explicit floats/lists
            results.append(
                {
                    "Distribution": str(name),
                    "AIC": float(aic),
                    "Params": [float(p) for p in params],
                    "NLLF": float(nllf),
                }
            )
        except Exception as e:
            print(f"Skipping {name}: {e}")

    # Explicitly define the schema to avoid inference errors
    schema = {
        "Distribution": pl.String,
        "AIC": pl.Float64,
        "Params": pl.List(pl.Float64),
        "NLLF": pl.Float64,
    }

    summary = pl.from_dicts(results, schema=schema).sort("AIC")
    return summary

In [ ]:
summary = find_best_distribution_constrained(df_rookies, "Grade")
print(summary)
params = {}
for distribution in summary.to_dicts():
    params[distribution["Distribution"]] = distribution["Params"]
params

In [ ]:
# 1. Your results mapped into a dictionary
# Using the values from your summary table
fits = {
    "Gamma": {"dist": stats.gamma, "params": params["Gamma"]},
    "Beta": {"dist": stats.beta, "params": params["Beta"]},
    "Log-Normal": {"dist": stats.lognorm, "params": params["Log-Normal"]},
    "Skew-Normal": {"dist": stats.skewnorm, "params": params["Skew-Normal"]},
}

# 2. Generate x-values for the curves
x = np.linspace(0, 100, 500)

# 3. Create a list of DataFrames (one for each distribution) then concat
pdf_frames = []
for name, info in fits.items():
    pdf_values = info["dist"].pdf(x, *info["params"])
    pdf_frames.append(pl.DataFrame({"Grade": x, "Density": pdf_values, "Distribution": name}))

curves_df = pl.concat(pdf_frames)

# 4. Plot the normalized histogram
hist = df_rookies.hvplot.hist(
    "Grade", bins="auto", normed=True, alpha=0.3, color="gray", label="Observed Data"
)

# 5. Plot the lines
lines = curves_df.hvplot.line(
    x="Grade",
    y="Density",
    by="Distribution",
    line_width=3,
    title="Comparison of Fitted Distributions",
)

# Overlay
hist * lines